# Example Dataset
For the remainder of this lesson, we will walk slowly through the process of building a multilevel/mixed-effects model for a single example data structure. This will be broken down into the individual steps described previously. Each step will contain a reasonable amount of exposition so that you can see how it is applied to this specific example. The hope is that, by the end, the process itself will be clearer. In the associated workshop, there will be multiple examples to choose from, so you can see the same steps and logic applied to other types of data structure.

## The Rat Pup Data
For this example, we will be using data on the birth weight of litters of rat pups. Of course, in the realm of psychology, you may never find yourself working with baby rats. However, it is useful to see many different examples from different fields so you get a sense of how these methods can be applied *generically*. Otherwise, it is quite easy to get stuck conceptualising these approaches in terms of `subjects` and nothing else.

### Data Description
The data consists of the birth weight of rat pups from mothers who were assigned to one of three different treatments. These were either a *high*-dose, a *low*-dose, or a *control* that received no treatment. Initially, 10 female rats were assigned to each treatment, but three of the rats in the high-dose treatment died. This means that both the low-dose treatments and control treatment consist of 10 litters, but the high-dose only consists of 7 litters. This makes the data *unbalanced* in terms of litters-per-treatment. In addition, each litter varied in the number of pups, with the lowest being 2 pups and the highest being 18 pups. So the data are also unbalanced in terms of pups-per-litter. The experimental question concern the effect of the treatments on the birth weights of the pups, as well as whether this effect relates to additional factors such as the sex of the pups.

### Downloading and Wrangling the Data
The data are available for download from [here](https://websites.umich.edu/~bwest/chapter3.html). Unfortunately, the website does not allow direct downloading of the data through `R`, so you will need to visit the site and download it directly into the current working directory. Below, we read the data into `R` using `read.table()` and then print the first 26 rows (corresponding to the first 2 litters).

In [4]:
ratpup <- read.table('rat_pup.dat', header=TRUE)
ratpup[1:26,]

   pup_id weight    sex litter litsize treatment
1       1   6.60   Male      1      12   Control
2       2   7.40   Male      1      12   Control
3       3   7.15   Male      1      12   Control
4       4   7.24   Male      1      12   Control
5       5   7.10   Male      1      12   Control
6       6   6.04   Male      1      12   Control
7       7   6.98   Male      1      12   Control
8       8   7.05   Male      1      12   Control
9       9   6.95 Female      1      12   Control
10     10   6.29 Female      1      12   Control
11     11   6.77 Female      1      12   Control
12     12   6.57 Female      1      12   Control
13     13   6.37   Male      2      14   Control
14     14   6.37   Male      2      14   Control
15     15   6.90   Male      2      14   Control
16     16   6.34   Male      2      14   Control
17     17   6.50   Male      2      14   Control
18     18   6.10   Male      2      14   Control
19     19   6.44   Male      2      14   Control
20     20   6.94   M

These data are already in *long-format*, so all we need to do is make sure that all categorical variables are correctly defined as a `factor` using

In [2]:
ratpup$sex       <- as.factor(ratpup$sex)
ratpup$litter    <- as.factor(ratpup$litter)
ratpup$treatment <- as.factor(ratpup$treatment)

Finally, we can briefly summarise the data and check that everything is as expected.

In [3]:
summary(ratpup)

     pup_id           weight          sex          litter       litsize     
 Min.   :  1.00   Min.   :3.680   Female:151   7      : 18   Min.   : 2.00  
 1st Qu.: 81.25   1st Qu.:5.650   Male  :171   8      : 17   1st Qu.:12.00  
 Median :161.50   Median :6.055                9      : 17   Median :14.00  
 Mean   :161.50   Mean   :6.081                11     : 16   Mean   :13.33  
 3rd Qu.:241.75   3rd Qu.:6.397                20     : 16   3rd Qu.:16.00  
 Max.   :322.00   Max.   :8.330                14     : 15   Max.   :18.00  
                                               (Other):223                  
   treatment  
 Control:131  
 High   : 65  
 Low    :126  
              
              
              
              

## Identifying the Structure
As a first step, we need to determine what *structure* we are dealing with. We ran through the process of doing this earlier, so this is simply an exercise in *applying* those steps. We include discussion of each step below for completeness.

### 1. Identify the Units of Analysis
To begin, we need to identify our *unit of analysis*. This is the element of the data that we are trying to *predict* or *model*. Based on the experimental question, we are interested in the birth weight of *rat pups*. So, in this example, our units are the *individual rat pups*. The primary question is about how the treatments affect the weight of the pups themselves.

### 2. Identify the Rows of the Data
In the second step, we need to determine what each row of the data represents. Let us go back to the data and print out the first few rows

In [4]:
head(ratpup)

  pup_id weight  sex litter litsize treatment
1      1   6.60 Male      1      12   Control
2      2   7.40 Male      1      12   Control
3      3   7.15 Male      1      12   Control
4      4   7.24 Male      1      12   Control
5      5   7.10 Male      1      12   Control
6      6   6.04 Male      1      12   Control

As we can see, each row corresponds to a *single unit* measured *once*. The birth weight of each pup was not taken multiple times, so these are *not* repeated measurements or longitudinal. So, these data are either *clustered* or there is *no dependency structure* and we go back to the normal linear model. 

### 3. Identify Clusters
In the final step, we need to determine if there are any clusters. At this point in our reasoning there *may* or may *not* be, and there may also be *several* nested clusters. Looking again at the first few rows of data

In [5]:
head(ratpup)

  pup_id weight  sex litter litsize treatment
1      1   6.60 Male      1      12   Control
2      2   7.40 Male      1      12   Control
3      3   7.15 Male      1      12   Control
4      4   7.24 Male      1      12   Control
5      5   7.10 Male      1      12   Control
6      6   6.04 Male      1      12   Control

we can see several variables that are constant across different pups: `sex`, `litter`, `litsize` and `treatment`. The important question is whether any of these represent a *shared latent influence* that could affect how correlated the measures of `weight` are between different pups? If we think back to our *sticky notes* vs *rubber bands* metaphor, do any of these elements tie units together or are they simply labels that have been given those units? 

We can also think back to the difference between something each unit *HAS* versus something each unit is *IN*. Each rat pup *HAS* a sex, treatment and litter size, but is *IN* a single litter. So, in this example, `sex`, `litsize` and `treatment` are all *sticky notes*. They are *labels* given to each individual unit. They describe a *characteristic* of the pup or a *characteristic* of their environment, but they do not tie them together. By comparison, `litter` itself is a *rubber band*. All pups from the same litter are tied together by the biological reality of sharing a mother, genes, siblings and rearing environment. So, our conclusion here, is that `litter` *is* a clustering variable. All pups from the same `litter` are therefore expected to be correlated to some degree.

Remember, we are trying to reason about whether some shared environment existed when the data were sampled that *could* make certain observations more similar than others. Clustering is all about how the data were collected. If we think about what elements were *independently selected*, the answer is that the *individual mother rats* were selected independently. These mother rats define the different litters. So the litters themselves are a random selection from all possible litters and the litters themselves are independent of each other. So, `litter` itself defines a *boundary of dependence*. Anything that lives *within* a litter may be correlated but anything that lives *across* litters should be independent.

Beyond litter, there are no higher-level clusters and so we conclude that we only have a *two-level* clustered dataset. However, for the purpose of illustration, we *could* imagine a similar experiment where each mother rat gave birth to *multiple* litters. In that situation, we would have measures of pup weights clustered within litters, but would also have litters clustered within mothers. This would then be an example of a *three-level* clustered dataset.

## Structure to Hierarchy
Now that we have determined that we have a dataset with a single clustering variable of `litter`, we can start thinking hierarchically about the data. This will provide the framework for building our multilevel model. If we look back on our definitions of the levels from earlier in the lesson, we had:

- **Level 1** represents observations at the *most detailed* level of the dataset. For *clustered* data, this represents the *units* of analysis.
- **Level 2** represents the next level in the hierarchy. For *clustered* data, this will be the *clusters of units*.
- **Level 3** represents the next level in the hierarchy, above Level 2. For *clustered* data, this will represent *clusters of clusters*.

So this tells us that, for this example, Level 1 represents the weight of *individual rat pups*. This is commensurate with the rows of the dataset identified earlier. Level 2 then represents the *litters* of rat pups, as these define the clusters of units. Level 3 does not exist for these data as we have no *clusters of clusters*.

Our final conclusion is therefore that we have a 2-level clustered dataset as follows


| **Data Type**    | **Clustered**    | 
|------------------|------------------|
| **Dataset**      | `Ratpup`         |
| **Level 1**      | _Individual Pup_ |
| **Level 2**      | Litter           |
| **Level 3**      | -                |

`````{topic} What do you now know?
In this section, we have discussed the example dataset we will be using for the model-building example. After reading this section, you should have a good sense of :

- The structure of the dataset, as well as the core experimental question.
- The idea that mixed-effects models still apply equally, even though we are dealing with *rats* instead of *humans*.
- How the process of structure-identification lead to the conclusions that: 
    - The *unit of analysis* is *rat pup*
    - The data are *clustered*, with *pups* inside *litters*
- How this structure can be translated into multilevel form, with Level 1 equaling the *individual pup* and Level 2 equaling each *litter*.
- What a *three-level* version of these data would look like.

`````